<a href="https://colab.research.google.com/github/janithcyapa/Wearable-Biomechanics/blob/main/Joint%20Angle%20Trial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **🏋🏽 Wearable Biomechanics - Joint Angle Trial**
> MARC - Multidisciplinary AI Research Centre | UOP


## **🛠️ Setup OpenDatasetStore**

In [1]:
%%capture
# @title Store Initialization
from google.colab import drive
import pandas as pd
import os
!pip install -q ultralytics opencv-python pandas scipy
import ultralytics
print("Ultralytics version:", ultralytics.__version__)

# Mount Google Drive
drive.mount('/content/drive', force_remount=True)

# Download the latest store manager
!pip install git+https://github.com/janithcyapa/OpenDatasetStore.git

from open_dataset_store import OpenDatasetStore

dataset_path = "/content/drive/MyDrive/Datasets/Wearable-Biomechanics"
store = OpenDatasetStore(
    base_dir=dataset_path,
    backend="local",
    entity_id_format="sub_{num:04d}",
    entry_id_format="{type}_{num:04d}",
    raw_filename_format="{entry_id}_{entity_id}_{ts}.csv",
    processed_filename_format="{entry_id}_{tag}.parquet"
)


In [ ]:
store.summary()
store.list_entries(entry_type="experiments")

# Visual Estimation

In [4]:
entry_id = "experiments_0001"

In [5]:
# @title Download Files
import os
import glob

entry_type = "experiments"
labels = ["front_video", "side_video"]

# Create a subfolder for the experiment
exp_folder = f"/content/{entry_id}"
os.makedirs(exp_folder, exist_ok=True)

print(f"Setting up directory: {exp_folder}")

downloaded_videos = []
for label in labels:
    print(f"Downloading {label}...")
    # Download into the designated subfolder
    store.download_related_file(
        entry_type=entry_type,
        entry_id=entry_id,
        label=label,
        dest_folder=exp_folder
    )

# Retrieve the paths of the downloaded mp4 files
downloaded_videos = glob.glob(f"{exp_folder}/*.mp4")
print("\nDownloaded videos:")
for vid in downloaded_videos:
    print(vid)

Setting up directory: /content/experiments_0001
📥 Downloaded to: /content/experiments_0001/experiments_0001_sub_0001_front_video.mp4
📥 Downloaded to: /content/experiments_0001/experiments_0001_sub_0001_side_video.mp4

Downloaded videos:
/content/experiments_0001/experiments_0001_sub_0001_front_video.mp4
/content/experiments_0001/experiments_0001_sub_0001_side_video.mp4


In [22]:
# @title Run Visual Estimation
import json
import cv2
import os
from ultralytics import YOLO

# Load the model
model = YOLO('yolo11x-pose.pt')

def extract_keypoints(video_path, output_json_path):
    cap = cv2.VideoCapture(video_path)
    all_frames_data = []
    frame_idx = 0

    print(f"Processing: {os.path.basename(video_path)}")
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        results = model(frame, verbose=False)
        frame_data = {"frame": frame_idx, "keypoints": [], "scores": []}

        if len(results) > 0 and results[0].keypoints is not None:
            keypoints = results[0].keypoints.xy[0].cpu().numpy().tolist()
            scores = results[0].keypoints.conf[0].cpu().numpy().tolist()

            frame_data["keypoints"] = keypoints
            frame_data["scores"] = scores

        all_frames_data.append(frame_data)
        frame_idx += 1

    cap.release()

    with open(output_json_path, "w") as f:
        json.dump(all_frames_data, f)
    print(f"Saved {frame_idx} frames to {output_json_path}")

# Run extraction on all downloaded feeds in the folder
json_files = []
for video_path in downloaded_videos:
    base_name = os.path.basename(video_path)
    json_name = base_name.replace('.mp4', '_keypoints.json')
    json_path = os.path.join(exp_folder, json_name)

    extract_keypoints(video_path, json_path)
    json_files.append(json_path)

Processing: experiments_0001_sub_0001_front_video.mp4
Saved 2375 frames to /content/experiments_0001/experiments_0001_sub_0001_front_video_keypoints.json
Processing: experiments_0001_sub_0001_side_video.mp4
Saved 2247 frames to /content/experiments_0001/experiments_0001_sub_0001_side_video_keypoints.json


In [ ]:
# @title
import json
import cv2
import os
import numpy as np
from base64 import b64encode
from IPython.display import HTML, display

# Standard COCO 17-keypoint skeleton connections
SKELETON_PAIRS = [
    (0, 1), (0, 2), (1, 3), (2, 4),
    (5, 6),
    (5, 7), (7, 9),
    (6, 8), (8, 10),
    (5, 11), (6, 12),
    (11, 12),
    (11, 13), (13, 15),
    (12, 14), (14, 16)
]
CONFIDENCE_THRESHOLD = 0.4

rendered_videos = []

for video_path in downloaded_videos:
    base_name = os.path.basename(video_path)
    json_path = os.path.join(exp_folder, base_name.replace('.mp4', '_keypoints.json'))

    # Change extension to .webm for open-source browser compatibility
    output_video_path = os.path.join(exp_folder, base_name.replace('.mp4', '_visualized.webm'))

    with open(json_path, "r") as f:
        keypoints_data = json.load(f)

    cap = cv2.VideoCapture(video_path)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)

    # Use VP80 codec which is natively supported by OpenCV and HTML5
    fourcc = cv2.VideoWriter_fourcc(*'VP80')
    out = cv2.VideoWriter(output_video_path, fourcc, fps, (width, height))

    frame_idx = 0
    print(f"\nRendering visual skeleton overlay for {base_name}...")
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret or frame_idx >= len(keypoints_data):
            break

        frame_info = keypoints_data[frame_idx]
        kpts = frame_info.get("keypoints", [])
        scores = frame_info.get("scores", [])

        if len(kpts) == 17:
            for p1, p2 in SKELETON_PAIRS:
                if scores[p1] >= CONFIDENCE_THRESHOLD and scores[p2] >= CONFIDENCE_THRESHOLD:
                    pt1 = (int(kpts[p1][0]), int(kpts[p1][1]))
                    pt2 = (int(kpts[p2][0]), int(kpts[p2][1]))
                    cv2.line(frame, pt1, pt2, (0, 255, 0), 2, lineType=cv2.LINE_AA)

            for i, (x, y) in enumerate(kpts):
                if scores[i] >= CONFIDENCE_THRESHOLD:
                    cv2.circle(frame, (int(x), int(y)), 4, (0, 0, 255), -1, lineType=cv2.LINE_AA)

        cv2.putText(frame, f"Frame: {frame_idx}", (25, 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 255, 255), 2, lineType=cv2.LINE_AA)

        out.write(frame)
        frame_idx += 1

    cap.release()
    out.release()
    rendered_videos.append(output_video_path)
    print(f"Visualization complete: {output_video_path}")


html_code = "<div style='display: flex; gap: 20px;'>"

for vid_path in rendered_videos:
    if os.path.exists(vid_path) and os.path.getsize(vid_path) > 0:
        try:
            webm_data = open(vid_path, 'rb').read()
            # Specify video/webm in the data URL
            data_url = "data:video/webm;base64," + b64encode(webm_data).decode()
            html_code += f"""
            <div>
                <h4 style="margin-bottom: 5px;">{os.path.basename(vid_path)}</h4>
                <video width=480 controls>
                      <source src="{data_url}" type="video/webm">
                </video>
            </div>
            """
        except Exception as e:
            print(f"Failed to load {vid_path}: {e}")
    else:
        print(f"File missing or empty: {vid_path}")

html_code += "</div>"
display(HTML(html_code))


Rendering visual skeleton overlay for experiments_0001_sub_0001_front_video.mp4...


In [6]:
# @title Fast Low-Res Visualization
import json
import cv2
import os
import numpy as np
from base64 import b64encode
from IPython.display import HTML, display

SKELETON_PAIRS = [
    (0, 1), (0, 2), (1, 3), (2, 4),
    (5, 6),
    (5, 7), (7, 9),
    (6, 8), (8, 10),
    (5, 11), (6, 12),
    (11, 12),
    (11, 13), (13, 15),
    (12, 14), (14, 16)
]
CONFIDENCE_THRESHOLD = 0.4
TARGET_WIDTH = 640  # Low resolution target for fast encoding

rendered_videos = []

for video_path in downloaded_videos:
    base_name = os.path.basename(video_path)
    json_path = os.path.join(exp_folder, base_name.replace('.mp4', '_keypoints.json'))
    output_video_path = os.path.join(exp_folder, base_name.replace('.mp4', '_visualized.webm'))

    with open(json_path, "r") as f:
        keypoints_data = json.load(f)

    cap = cv2.VideoCapture(video_path)
    orig_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    orig_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps = cap.get(cv2.CAP_PROP_FPS)

    # Calculate scale factors
    scale = TARGET_WIDTH / float(orig_width)
    new_width = int(orig_width * scale)
    new_height = int(orig_height * scale)

    fourcc = cv2.VideoWriter_fourcc(*'VP80')
    out = cv2.VideoWriter(output_video_path, fourcc, fps, (new_width, new_height))

    frame_idx = 0
    print(f"\nRendering fast low-res overlay for {base_name}...")
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret or frame_idx >= len(keypoints_data):
            break

        # Resize frame immediately to reduce drawing and encoding overhead
        frame = cv2.resize(frame, (new_width, new_height))

        frame_info = keypoints_data[frame_idx]
        kpts = frame_info.get("keypoints", [])
        scores = frame_info.get("scores", [])

        if len(kpts) == 17:
            for p1, p2 in SKELETON_PAIRS:
                if scores[p1] >= CONFIDENCE_THRESHOLD and scores[p2] >= CONFIDENCE_THRESHOLD:
                    # Apply scale to coordinates
                    pt1 = (int(kpts[p1][0] * scale), int(kpts[p1][1] * scale))
                    pt2 = (int(kpts[p2][0] * scale), int(kpts[p2][1] * scale))
                    cv2.line(frame, pt1, pt2, (0, 255, 0), 2, lineType=cv2.LINE_AA)

            for i, (x, y) in enumerate(kpts):
                if scores[i] >= CONFIDENCE_THRESHOLD:
                    cv2.circle(frame, (int(x * scale), int(y * scale)), 3, (0, 0, 255), -1, lineType=cv2.LINE_AA)

        cv2.putText(frame, f"Frame: {frame_idx}", (15, 25),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2, lineType=cv2.LINE_AA)

        out.write(frame)
        frame_idx += 1

    cap.release()
    out.release()
    rendered_videos.append(output_video_path)
    print(f"Visualization complete: {output_video_path}")

html_code = "<div style='display: flex; gap: 20px;'>"

for vid_path in rendered_videos:
    if os.path.exists(vid_path) and os.path.getsize(vid_path) > 0:
        try:
            webm_data = open(vid_path, 'rb').read()
            data_url = "data:video/webm;base64," + b64encode(webm_data).decode()
            html_code += f"""
            <div>
                <h4 style="margin-bottom: 5px;">{os.path.basename(vid_path)}</h4>
                <video width={new_width} controls>
                      <source src="{data_url}" type="video/webm">
                </video>
            </div>
            """
        except Exception as e:
            print(f"Failed to load {vid_path}: {e}")
    else:
        print(f"File missing or empty: {vid_path}")

html_code += "</div>"
display(HTML(html_code))


Rendering fast low-res overlay for experiments_0001_sub_0001_front_video.mp4...
Visualization complete: /content/experiments_0001/experiments_0001_sub_0001_front_video_visualized.webm

Rendering fast low-res overlay for experiments_0001_sub_0001_side_video.mp4...
Visualization complete: /content/experiments_0001/experiments_0001_sub_0001_side_video_visualized.webm
Buffered data was truncated after reaching the output size limit.

In [8]:
import plotly.io as pio
pio.renderers.default = "colab"

In [9]:
# @title Plot Keypoint Coordinates
import json
import os
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Get JSON paths based on the downloaded videos
json_paths = []
for vid in downloaded_videos:
    base = os.path.basename(vid)
    json_path = os.path.join(exp_folder, base.replace('.mp4', '_keypoints.json'))
    if os.path.exists(json_path):
        json_paths.append(json_path)

if len(json_paths) == 2:
    # Create a 2x2 subplot layout
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=(
            f"{os.path.basename(json_paths[0])} - X",
            f"{os.path.basename(json_paths[0])} - Y",
            f"{os.path.basename(json_paths[1])} - X",
            f"{os.path.basename(json_paths[1])} - Y"
        )
    )

    # Loop through the two JSON files
    for i, j_path in enumerate(json_paths):
        with open(j_path, 'r') as f:
            data = json.load(f)

        frames = [d['frame'] for d in data]

        # Plot each of the 17 keypoints
        for kpt_idx in range(17):
            x_coords = []
            y_coords = []
            for d in data:
                # Check if keypoints were detected in this frame
                if len(d.get('keypoints', [])) == 17:
                    x_coords.append(d['keypoints'][kpt_idx][0])
                    y_coords.append(d['keypoints'][kpt_idx][1])
                else:
                    x_coords.append(None)
                    y_coords.append(None)

            # Add X trace (Row i+1, Col 1)
            fig.add_trace(
                go.Scatter(x=frames, y=x_coords, mode='lines', name=f'Keypoint {kpt_idx}',
                           showlegend=(i == 0)),
                row=i+1, col=1
            )

            # Add Y trace (Row i+1, Col 2)
            fig.add_trace(
                go.Scatter(x=frames, y=y_coords, mode='lines', name=f'Keypoint {kpt_idx}',
                           showlegend=False),
                row=i+1, col=2
            )

    # Update layout with the dark theme and sizing
    fig.update_layout(
        height=800,
        width=1200,
        title_text="Keypoint Coordinates over Time (X and Y)",
        template="plotly_dark",
        hovermode="x unified"
    )
    fig.show()
else:
    print(f"Expected 2 JSON files, but found {len(json_paths)}.")

In [10]:
# @title Synchronize Videos using Keypoint Movement
import json
import os
import numpy as np
from scipy import signal
import plotly.graph_objects as go

if len(json_paths) == 2:
    def get_movement_signal(json_path):
        with open(json_path, 'r') as f:
            data = json.load(f)

        # Extract all keypoints into a numpy array (frames, 17, 2)
        coords = []
        for d in data:
            kpts = d.get('keypoints', [])
            if len(kpts) == 17:
                coords.append(kpts)
            else:
                # If missing, copy the previous frame's keypoints to avoid spikes
                if len(coords) > 0:
                    coords.append(coords[-1])
                else:
                    coords.append([[0, 0]] * 17)
        coords = np.array(coords)

        # Calculate frame-to-frame difference (velocity)
        diffs = np.diff(coords, axis=0)

        # Calculate magnitude of velocity for each keypoint, then sum across all keypoints
        movement = np.sum(np.linalg.norm(diffs, axis=2), axis=1)
        return movement

    # Generate signals for both videos
    sig1 = get_movement_signal(json_paths[0])
    sig2 = get_movement_signal(json_paths[1])

    # Normalize signals for better cross-correlation
    sig1_norm = (sig1 - np.mean(sig1)) / (np.std(sig1) + 1e-8)
    sig2_norm = (sig2 - np.mean(sig2)) / (np.std(sig2) + 1e-8)

    # Perform cross-correlation to find the time delay
    correlation = signal.correlate(sig1_norm, sig2_norm, mode='full')
    lags = signal.correlation_lags(len(sig1_norm), len(sig2_norm), mode='full')

    best_lag = lags[np.argmax(correlation)]

    vid1_name = os.path.basename(json_paths[0]).replace('_keypoints.json', '')
    vid2_name = os.path.basename(json_paths[1]).replace('_keypoints.json', '')

    print(f"--- Synchronization Results ---")
    print(f"Calculated frame offset (lag): {best_lag} frames")
    if best_lag > 0:
        print(f"{vid1_name} is ahead of {vid2_name} by {best_lag} frames.")
    elif best_lag < 0:
        print(f"{vid2_name} is ahead of {vid1_name} by {-best_lag} frames.")
    else:
        print("The videos are perfectly synchronized (0 frame lag).")

    # Plot the aligned signals for visual verification
    fig = go.Figure()

    if best_lag > 0:
        x1 = np.arange(len(sig1))
        x2 = np.arange(len(sig2)) + best_lag
    else:
        x1 = np.arange(len(sig1)) - best_lag
        x2 = np.arange(len(sig2))

    fig.add_trace(go.Scatter(x=x1, y=sig1_norm, mode='lines', name=f'{vid1_name} Movement', opacity=0.8))
    fig.add_trace(go.Scatter(x=x2, y=sig2_norm, mode='lines', name=f'{vid2_name} Movement', opacity=0.8))

    fig.update_layout(
        title=f"Aligned Movement Signals (Offset: {best_lag} frames)",
        xaxis_title="Aligned Frame Index",
        yaxis_title="Normalized Movement Magnitude",
        template="plotly_dark",
        hovermode="x unified",
        height=500
    )
    fig.show()
else:
    print(f"Expected exactly 2 JSON files to synchronize, but found {len(json_paths)}.")

--- Synchronization Results ---
Calculated frame offset (lag): 89 frames
experiments_0001_sub_0001_front_video is ahead of experiments_0001_sub_0001_side_video by 89 frames.


In [11]:
# @title Visual Verification of Synchronization
import cv2
import numpy as np
import os
from IPython.display import HTML, display
from base64 import b64encode

print("Generating synchronized side-by-side preview...")
vid1_path = downloaded_videos[0]
vid2_path = downloaded_videos[1]

cap1 = cv2.VideoCapture(vid1_path)
cap2 = cv2.VideoCapture(vid2_path)

# Apply the calculated frame offset
if best_lag > 0:
    # Video 1 is ahead, so we skip 'best_lag' frames in Video 1
    cap1.set(cv2.CAP_PROP_POS_FRAMES, best_lag)
elif best_lag < 0:
    # Video 2 is ahead
    cap2.set(cv2.CAP_PROP_POS_FRAMES, -best_lag)

fps = min(cap1.get(cv2.CAP_PROP_FPS), cap2.get(cv2.CAP_PROP_FPS))
if fps == 0: fps = 30.0

target_h = 480
ret1, f1 = cap1.read()
ret2, f2 = cap2.read()

if ret1 and ret2:
    w1 = int(f1.shape[1] * (target_h / f1.shape[0]))
    w2 = int(f2.shape[1] * (target_h / f2.shape[0]))
    out_w = w1 + w2

    # Reset positions to start of sync
    cap1.set(cv2.CAP_PROP_POS_FRAMES, best_lag if best_lag > 0 else 0)
    cap2.set(cv2.CAP_PROP_POS_FRAMES, -best_lag if best_lag < 0 else 0)

    out_path = os.path.join(exp_folder, 'synchronized_playback.webm')
    fourcc = cv2.VideoWriter_fourcc(*'VP80')
    out = cv2.VideoWriter(out_path, fourcc, fps, (out_w, target_h))

    frame_count = 0
    # Render a preview clip (e.g., up to 600 frames / ~20 seconds) to avoid long processing
    MAX_FRAMES = 600

    while frame_count < MAX_FRAMES:
        ret1, frame1 = cap1.read()
        ret2, frame2 = cap2.read()
        if not ret1 or not ret2:
            break

        f1_resized = cv2.resize(frame1, (w1, target_h))
        f2_resized = cv2.resize(frame2, (w2, target_h))

        # Draw original frame numbers for verification
        cv2.putText(f1_resized, f"{vid1_name} Frame: {int(cap1.get(cv2.CAP_PROP_POS_FRAMES))}", (15, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
        cv2.putText(f2_resized, f"{vid2_name} Frame: {int(cap2.get(cv2.CAP_PROP_POS_FRAMES))}", (15, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

        combined = np.hstack((f1_resized, f2_resized))
        out.write(combined)
        frame_count += 1

    cap1.release()
    cap2.release()
    out.release()
    print(f"Saved synchronized video to {out_path} ({frame_count} frames)\n")

    # Display the result
    if os.path.exists(out_path) and os.path.getsize(out_path) > 0:
        webm_data = open(out_path, 'rb').read()
        data_url = "data:video/webm;base64," + b64encode(webm_data).decode()
        html = f'''
        <h4 style="margin-bottom: 5px;">Synchronized Playback Preview (First {MAX_FRAMES} frames)</h4>
        <video width="{out_w}" controls>
              <source src="{data_url}" type="video/webm">
        </video>
        '''
        display(HTML(html))
else:
    print("Failed to read videos.")

Generating synchronized side-by-side preview...
Saved synchronized video to /content/experiments_0001/synchronized_playback.webm (600 frames)



In [12]:
# @title Generate Complete Full-Size Synchronized Video
import cv2
import numpy as np
import os

print("Generating COMPLETE synchronized side-by-side video at FULL resolution...")
print("NOTE: This will take some time. The output file will not be previewed inline to avoid crashing the browser.")

vid1_path = downloaded_videos[0]
vid2_path = downloaded_videos[1]

cap1 = cv2.VideoCapture(vid1_path)
cap2 = cv2.VideoCapture(vid2_path)

# Apply the calculated frame offset
if best_lag > 0:
    cap1.set(cv2.CAP_PROP_POS_FRAMES, best_lag)
elif best_lag < 0:
    cap2.set(cv2.CAP_PROP_POS_FRAMES, -best_lag)

fps = min(cap1.get(cv2.CAP_PROP_FPS), cap2.get(cv2.CAP_PROP_FPS))
if fps == 0: fps = 30.0

ret1, f1 = cap1.read()
ret2, f2 = cap2.read()

if ret1 and ret2:
    # Use the original height of the first video
    target_h = f1.shape[0]
    w1 = f1.shape[1]
    w2 = int(f2.shape[1] * (target_h / f2.shape[0]))
    out_w = w1 + w2

    # Reset positions to start of sync
    cap1.set(cv2.CAP_PROP_POS_FRAMES, best_lag if best_lag > 0 else 0)
    cap2.set(cv2.CAP_PROP_POS_FRAMES, -best_lag if best_lag < 0 else 0)

    out_path = os.path.join(exp_folder, 'synchronized_playback_full.mp4')
    # mp4v is generally faster for high resolution encoding compared to VP8
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(out_path, fourcc, fps, (out_w, target_h))

    frame_count = 0
    total_frames_expected = int(min(
        cap1.get(cv2.CAP_PROP_FRAME_COUNT) - (best_lag if best_lag > 0 else 0),
        cap2.get(cv2.CAP_PROP_FRAME_COUNT) - (-best_lag if best_lag < 0 else 0)
    ))

    while True:
        ret1, frame1 = cap1.read()
        ret2, frame2 = cap2.read()
        if not ret1 or not ret2:
            break

        # Resize frame2 only if their heights differ to maintain aspect ratio side-by-side
        if frame1.shape[0] != frame2.shape[0]:
            f2_resized = cv2.resize(frame2, (w2, target_h))
        else:
            f2_resized = frame2

        f1_resized = frame1

        # Draw original frame numbers for verification (scaled up for high-res visibility)
        cv2.putText(f1_resized, f"{vid1_name} Frame: {int(cap1.get(cv2.CAP_PROP_POS_FRAMES))}", (50, 100), cv2.FONT_HERSHEY_SIMPLEX, 2.0, (0, 255, 0), 4)
        cv2.putText(f2_resized, f"{vid2_name} Frame: {int(cap2.get(cv2.CAP_PROP_POS_FRAMES))}", (50, 100), cv2.FONT_HERSHEY_SIMPLEX, 2.0, (0, 255, 0), 4)

        combined = np.hstack((f1_resized, f2_resized))
        out.write(combined)
        frame_count += 1

        if frame_count % 200 == 0:
            print(f"Processed {frame_count}/{total_frames_expected} frames...")

    cap1.release()
    cap2.release()
    out.release()
    print(f"\nSuccess! Saved full synchronized video to {out_path} ({frame_count} frames)")
    print("Please download the file directly from the Colab file browser panel on the left to view it.")
else:
    print("Failed to read videos.")

Generating COMPLETE synchronized side-by-side video at FULL resolution...
NOTE: This will take some time. The output file will not be previewed inline to avoid crashing the browser.
Processed 200/2247 frames...
Processed 400/2247 frames...
Processed 600/2247 frames...
Processed 800/2247 frames...
Processed 1000/2247 frames...
Processed 1200/2247 frames...
Processed 1400/2247 frames...
Processed 1600/2247 frames...
Processed 1800/2247 frames...
Processed 2000/2247 frames...
Processed 2200/2247 frames...

Success! Saved full synchronized video to /content/experiments_0001/synchronized_playback_full.mp4 (2247 frames)
Please download the file directly from the Colab file browser panel on the left to view it.


In [13]:
# Replace with your actual file name and desired Drive folder
!cp /content/experiments_0001/synchronized_playback_full.mp4 /content/drive/MyDrive/